# LLM Client Experiments
This notebook demonstrates how to use the custom LLM client with OpenAI and Together AI providers.

In [ ]:
# Setup path to import the llm_client module
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

## Initialize the LLM Client

In [ ]:
from llm_client.client import LLMClient
from llm_client.config import LLMConfig

# Initialize with default config (reads from environment variables)
client = LLMClient()

# Or create a custom config
# config = LLMConfig(
#     openai_api_key="your-api-key",
#     openai_model="gpt-4",
#     together_api_key="your-together-key",
#     together_model="meta-llama/Llama-2-70b-chat-hf"
# )
# client = LLMClient(config)

## Simple Chat Example

In [ ]:
# Simple chat with the model
response = client.chat(
    message="What is the capital of France?",
    temperature=0.7
)
print(response)

## Chat with System Prompt

In [ ]:
# Chat with a system prompt
response = client.chat(
    message="Tell me about Python",
    system_prompt="You are a helpful programming assistant. Keep your responses concise and technical.",
    temperature=0.5,
    max_tokens=150
)
print(response)

## Multi-turn Conversation

In [ ]:
# Multi-turn conversation using the complete method
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed."},
    {"role": "user", "content": "Can you give me a simple example?"}
]

response = client.complete(
    messages=messages,
    temperature=0.7,
    max_tokens=200
)

print(f"Response: {response.content}")
print(f"\nModel: {response.model}")
print(f"Provider: {response.provider}")
if response.usage:
    print(f"Tokens used: {response.usage}")

## Test Fallback Behavior

In [ ]:
# Test with fallback disabled
try:
    response = client.complete(
        messages="Hello, how are you?",
        use_fallback=False  # Only use primary provider
    )
    print(f"Primary provider response: {response.content}")
except Exception as e:
    print(f"Primary provider failed: {e}")

# Test with fallback enabled (default)
try:
    response = client.complete(
        messages="Hello, how are you?",
        use_fallback=True  # Use fallback if primary fails
    )
    print(f"Response (with fallback): {response.content}")
    print(f"Used provider: {response.provider}")
except Exception as e:
    print(f"All providers failed: {e}")

## Streaming Response (if supported)

In [ ]:
# Stream response (prints tokens as they arrive)
print("Streaming response:")
response = client.complete(
    messages="Write a short poem about programming",
    temperature=0.8,
    stream=True  # Enable streaming
)
print(f"\n\nFull response saved: {response.content[:100]}...")

## Check Provider Availability

In [ ]:
# Check which providers are available
if client.primary_provider:
    print(f"Primary provider available: {client.primary_provider.is_available()}")
    print(f"Primary model: {client.config.openai_model}")
else:
    print("No primary provider configured")

if client.fallback_provider:
    print(f"Fallback provider available: {client.fallback_provider.is_available()}")
    print(f"Fallback model: {client.config.together_model}")
else:
    print("No fallback provider configured")

## Custom Parameters Example

In [ ]:
# Pass custom parameters to the underlying provider
response = client.complete(
    messages="Generate a list of 3 creative project ideas",
    temperature=0.9,
    max_tokens=150,
    top_p=0.95,  # Custom parameter
    frequency_penalty=0.5  # Custom parameter
)
print(response.content)